[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/30_sequence_models/02_vanishing_exploding_gradients.ipynb)

# Bài 02 — Vanishing và exploding gradient trong RNN

**🎯 Mục tiêu**
- Vì sao gradient qua thời gian là **tích** của nhiều ma trận Jacobian; vai trò của **bán kính phổ** $\rho(W_{hh})$ và đạo hàm $\tanh$.
- Đo trực tiếp: $\lVert\partial\mathbf h_T/\partial\mathbf h_0\rVert$ theo $T$; gradient tới từng bước đầu vào ở RNN vs LSTM.
- **Exploding gradient** và **gradient clipping**; bảng các biện pháp.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/30_sequence_models

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from ml_course.visualization import set_style

try:
    import torch
    from torch import nn
except ImportError as e:
    raise ImportError("Chương này cần PyTorch: pip install -e \".[dl]\"") from e

set_style()

## 1. Gradient qua thời gian là một tích

Với $\mathbf h_t = \tanh(W_{hh}\mathbf h_{t-1} + W_{xh}\mathbf x_t + \mathbf b)$:
$$\frac{\partial\mathbf h_T}{\partial\mathbf h_k} = \prod_{t=k+1}^{T}\underbrace{\text{diag}\big(1 - \mathbf h_t^2\big)\,W_{hh}}_{J_t}$$
Tín hiệu lỗi từ bước $T$ về bước $k$ phải đi qua $T - k$ ma trận $J_t$. Nếu "độ lớn" mỗi $J_t$ nhỏ hơn 1 → tích **tắt dần theo hàm mũ**; lớn hơn 1 → **bùng nổ**. Với phần tuyến tính, độ lớn đó do **bán kính phổ** $\rho(W_{hh})$ (trị riêng lớn nhất về trị tuyệt đối) quyết định; $\tanh' \le 1$ chỉ làm nó nhỏ thêm.

### Trường hợp tuyến tính: $W^{T}$

In [ ]:
rng = np.random.default_rng(0)
H = 64
W_base = rng.normal(0, 1, (H, H))
W_base /= np.max(np.abs(np.linalg.eigvals(W_base)))                     # chuẩn hoá bán kính phổ = 1
Ts = np.arange(1, 101)
fig, ax = plt.subplots(figsize=(9, 4))
for rho in [0.9, 1.0, 1.1]:
    W = rho * W_base
    norms, P = [], np.eye(H)
    for _ in Ts:
        P = W @ P
        norms.append(np.linalg.norm(P, 2))
    ax.semilogy(Ts, norms, label=f"ρ = {rho}")
    print(f"ρ = {rho}: ‖W^T‖ tại T = 10 / 50 / 100 = {norms[9]:.2e} / {norms[49]:.2e} / {norms[99]:.2e}")
ax.set_xlabel("số bước T")
ax.set_ylabel("‖W^T‖ (log)")
ax.legend()
plt.show()

Chỉ cần $\rho$ lệch khỏi 1 chút, sau 100 bước gradient nhỏ đi hoặc lớn lên hàng nghìn lần.

## 2. RNN tanh thật: $\lVert\partial\mathbf h_T/\partial\mathbf h_0\rVert$

Dùng autograd đo trực tiếp với RNN tanh, trọng số ngẫu nhiên có bán kính phổ khác nhau, đầu vào ngẫu nhiên:

In [ ]:
def jacobian_norms(rho, T=80, H=64, seed=0):
    g = torch.Generator().manual_seed(seed)
    W = torch.randn(H, H, generator=g)
    W = W / torch.linalg.eigvals(W).abs().max() * rho
    U = torch.randn(H, 8, generator=g) * 0.5
    xs = torch.randn(T, 8, generator=g)
    h0 = torch.zeros(H, requires_grad=True)
    h, out = h0, []
    for t in range(T):
        h = torch.tanh(W @ h + U @ xs[t])
        out.append(h)
    norms = []
    for t in [1, 5, 10, 20, 40, 80]:
        grad = torch.autograd.grad(out[t - 1].sum(), h0, retain_graph=True)[0]   # ∂(Σ h_t)/∂h_0
        norms.append(grad.norm().item())
    return norms


for rho in [0.5, 1.0, 2.0, 4.0]:
    print(f"ρ = {rho}: ‖∂(Σh_T)/∂h_0‖ với T = 1, 5, 10, 20, 40, 80 → {np.array(jacobian_norms(rho)).round(4)}")

- $\rho = 0.5$ và $1$: gradient về $\mathbf h_0$ tắt về 0 chỉ sau 10–20 bước — mạng không thể học phụ thuộc xa.
- $\rho = 2$: phần tuyến tính phóng đại, nhưng $\mathbf h_t$ bị đẩy vào vùng **bão hoà** của tanh ($1 - h^2 \approx 0$) → gradient giữ được lâu hơn rồi **vẫn tắt dần**.
- $\rho = 4$: phóng đại thắng bão hoà → gradient **bùng nổ** (tăng hàng chục bậc độ lớn).

RNN tanh bị kẹp giữa hai đầu — tắt dần hoặc bùng nổ, rất khó có "vùng an toàn" cho chuỗi dài.

## 3. Gradient tới từng bước đầu vào: RNN vs LSTM

Một ví dụ chuỗi dài 100 bước, loss chỉ phụ thuộc đầu ra cuối. Đo $\lVert\partial L/\partial\mathbf x_t\rVert$ — bước $t$ càng xa cuối, gradient tới nó càng yếu?

In [ ]:
torch.manual_seed(0)
T, D, Hd = 100, 8, 64
x = torch.randn(1, T, D, requires_grad=True)
models = {"RNN (tanh)": nn.RNN(D, Hd, batch_first=True), "LSTM": nn.LSTM(D, Hd, batch_first=True), "GRU": nn.GRU(D, Hd, batch_first=True)}
with torch.no_grad():                                                   # cổng quên LSTM (thứ tự i, f, g, o) và cổng cập nhật GRU (r, z, n): bias = 1 → mặc định "giữ trí nhớ"
    models["LSTM"].bias_ih_l0[Hd:2 * Hd].fill_(1.0)
    models["GRU"].bias_ih_l0[Hd:2 * Hd].fill_(1.0)
fig, ax = plt.subplots(figsize=(10, 4))
for name, model in models.items():
    if x.grad is not None:
        x.grad = None
    out, _ = model(x)
    out[0, -1].sum().backward()
    gnorm = x.grad[0].norm(dim=1).numpy()
    ax.semilogy(range(1, T + 1), gnorm, label=name)
    print(f"{name:11s}: ‖∂L/∂x_t‖ tại t = 100 / 90 / 50 / 1 = {gnorm[99]:.1e} / {gnorm[89]:.1e} / {gnorm[49]:.1e} / {gnorm[0]:.1e}")
ax.set_xlabel("bước đầu vào t (loss tính ở bước 100)")
ax.set_ylabel("‖∂L/∂x_t‖ (log)")
ax.legend()
plt.show()

Với RNN, gradient tới các bước đầu nhỏ hơn bước cuối nhiều bậc độ lớn — thông tin ở đầu chuỗi gần như **không thể** ảnh hưởng tới việc học. LSTM/GRU (bài 03) có đường đi **cộng** cho trạng thái ô ($\mathbf c_t = \mathbf f_t\odot\mathbf c_{t-1} + \dots$) với cổng quên gần 1 → gradient suy giảm chậm hơn hẳn.

## 4. Exploding gradient và gradient clipping

Khi gradient **bùng nổ**, một bước cập nhật quá lớn đẩy tham số đi xa → loss nhảy vọt hoặc NaN. **Gradient clipping** (Pascanu et al., 2013): nếu $\lVert\mathbf g\rVert > c$ thì $\mathbf g \leftarrow c\,\mathbf g/\lVert\mathbf g\rVert$ — giữ **hướng**, giới hạn **độ dài** bước.

Thử nghiệm: RNN khởi tạo với $\rho(W_{hh}) = 3$ (dễ bùng nổ), SGD learning rate 0.2, nhiệm vụ hồi quy đơn giản trên chuỗi 30 bước:

In [ ]:
def exploding_run(clip, steps=300, seed=0, lr=0.2):
    torch.manual_seed(seed)
    rnn = nn.RNN(4, 32, batch_first=True)
    head = nn.Linear(32, 1)
    with torch.no_grad():
        W = rnn.weight_hh_l0
        W.copy_(W / torch.linalg.eigvals(W).abs().max() * 3.0)
    params = list(rnn.parameters()) + list(head.parameters())
    opt = torch.optim.SGD(params, lr=lr)
    g = torch.Generator().manual_seed(seed)
    losses, grad_norms = [], []
    for _ in range(steps):
        xb = torch.randn(32, 30, 4, generator=g)
        yb = xb[:, :, 0].mean(1, keepdim=True)                         # mục tiêu: trung bình kênh 0
        out, _ = rnn(xb)
        loss = nn.functional.mse_loss(head(out[:, -1]), yb)
        opt.zero_grad()
        loss.backward()
        total = torch.sqrt(sum((p.grad ** 2).sum() for p in params)).item()
        if clip is not None:
            nn.utils.clip_grad_norm_(params, clip)
        opt.step()
        losses.append(loss.item())
        grad_norms.append(total)
    return np.array(losses), np.array(grad_norms)


fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for clip in [None, 1.0]:
    losses, gnorms = exploding_run(clip)
    label = "không clipping" if clip is None else f"clip ‖g‖ ≤ {clip}"
    axes[0].semilogy(np.nan_to_num(losses, nan=1e6, posinf=1e6), label=label)
    axes[1].semilogy(np.nan_to_num(gnorms, nan=1e6, posinf=1e6), label=label)
    finite = np.isfinite(losses)
    first_nan = int(np.argmax(~finite)) if not finite.all() else None
    status = f"NaN từ bước {first_nan}" if first_nan is not None else f"loss trung bình 50 bước cuối = {np.mean(losses[-50:]):.4f}"
    print(f"{label:16s}: {status}")
axes[0].set_title("loss")
axes[1].set_title("chuẩn gradient (trước khi clip)")
for ax in axes:
    ax.legend()
    ax.set_xlabel("bước")
plt.show()

Không clipping, một bước gặp gradient khổng lồ đẩy tham số đi xa → loss bùng nổ thành NaN và không bao giờ hồi phục. Với clipping, mỗi bước bị giới hạn độ dài → huấn luyện **không bùng nổ**. Nhưng loss vẫn cao (so với ≈ 0.033 nếu chỉ dự đoán hằng số 0 — phương sai của trung bình 30 số chuẩn): khởi tạo $\rho = 3$ và learning rate 0.2 vẫn là thiết lập tồi. Clipping là **lưới an toàn** rẻ (gần như luôn bật khi train RNN), không thay thế khởi tạo và learning rate hợp lý — với $\rho \approx 1$ và lr nhỏ hơn, cùng mô hình học tốt. Clipping cũng **không** chữa vanishing gradient — nó chỉ chặn đầu bùng nổ.

## 5. Các biện pháp

| Vấn đề | Biện pháp |
|---|---|
| Exploding | gradient clipping; learning rate nhỏ hơn; khởi tạo $W_{hh}$ trực giao/bán kính phổ ≈ 1 |
| Vanishing | **LSTM/GRU** (cổng, đường cộng — bài 03); khởi tạo đơn vị/trực giao (IRNN); kết nối tắt theo thời gian; LayerNorm |
| Phụ thuộc rất xa | **attention** (bài 04) và Transformer (chương 31): kết nối trực tiếp mọi cặp vị trí, đường gradient dài $O(1)$ thay vì $O(T)$ |
| Chuỗi quá dài để BPTT | truncated BPTT (bài 01) — đổi lại không học được phụ thuộc dài hơn cửa sổ |

## 🧠 Kiểm tra nhanh

1. Vì sao gradient qua thời gian là tích của nhiều ma trận? Bán kính phổ ảnh hưởng thế nào?
2. Vì sao tăng $\rho(W_{hh})$ không giải quyết được vanishing gradient với tanh?
3. Gradient clipping làm gì? Chữa được vanishing gradient không?
4. Vì sao LSTM/GRU giữ gradient tốt hơn?
5. Attention giúp gì với phụ thuộc xa?

## 📌 Tóm tắt

- $\partial\mathbf h_T/\partial\mathbf h_k$ = tích $T - k$ Jacobian → tắt dần ($\rho < 1$, bão hoà tanh) hoặc bùng nổ ($\rho > 1$).
- Đo được: RNN gần như không truyền gradient tới đầu chuỗi dài; LSTM/GRU tốt hơn nhiều.
- Clipping cho exploding; cổng/attention cho vanishing.

## 📚 Tài liệu tham khảo

- Bengio, Simard & Frasconi (1994) — *Learning Long-Term Dependencies with Gradient Descent is Difficult*
- Pascanu, Mikolov & Bengio (2013) — *On the difficulty of training recurrent neural networks*
- Le, Jaitly & Hinton (2015) — *A Simple Way to Initialize Recurrent Networks of Rectified Linear Units* (IRNN)